In [1]:
import os
import sys
import json
import time
import signal
import subprocess
from pathlib import Path

REPO = "https://github.com/eslam-ahmed43/llm-lab.git"
ROOT = Path("/kaggle/working/llm-lab")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
(ROOT / "logs").mkdir(exist_ok=True)
assert (ROOT / "src/exp5_internals.py").exists(), "exp5_internals.py is missing: push it to GitHub first"

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log --oneline -1; ls src", shell=True, capture_output=True, text=True).stdout)
print(sys.version)

Cloning into '/kaggle/working/llm-lab'...


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

98acc30 Exp 5: fix fp16 attention overflow, add divergence diagnostics
bench_inference.py
exp5_internals.py
hf_batched_server.py
hf_server.py
plot_lora.py
quantize.py
train_lora.py

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
%%bash
set -e
pip uninstall -y -q torchao || true
pip install -q peft accelerate
python -c "
import peft, transformers, torch
print('peft', peft.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-1.5B-Instruct')
print('model cached')
"

peft 0.20.0 | transformers 5.16.1 | torch 2.11.0+cu128
model cached


Fetching 10 files: 100%|██████████| 10/10 [00:26<00:00,  2.62s/it]


In [3]:
def run_cmd(tag, args, timeout_s=1800):
    log_path = ROOT / "logs" / f"exp5_{tag}.log"
    cmd = [sys.executable, "-X", "faulthandler", "-u", str(ROOT / "src/exp5_internals.py")] + args
    env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}

    def new_lines(shown):
        lines = log_path.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.strip() and "Loading weights" not in line and "unauthenticated" not in line:
                print(line[:400], flush=True)
        return len(lines)

    start, shown = time.time(), 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(ROOT))
        while proc.poll() is None:
            time.sleep(5)
            shown = new_lines(shown)
            if time.time() - start > timeout_s:
                os.kill(proc.pid, signal.SIGABRT)
                time.sleep(3)
                proc.kill()
                print(log_path.read_text(errors="replace")[-3000:])
                raise TimeoutError(f"{tag} exceeded {timeout_s}s")
    new_lines(shown)
    if proc.returncode != 0:
        print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{tag} failed with exit code {proc.returncode}")

In [4]:
run_cmd("selftest", ["selftest", "--out", str(ROOT / "results/internals_selftest.json")])

manual {'max_abs_diff': 1.79e-06, 'mean_abs_diff': 2.5e-07, 'ref_abs_max': 3.09620428, 'top1_agreement': 1.0, 'cache_vs_full_max_abs_diff': 1.31e-06, 'greedy_identical': True}
sdpa {'max_abs_diff': 0.0, 'mean_abs_diff': 0.0, 'ref_abs_max': 3.09620428, 'top1_agreement': 1.0, 'cache_vs_full_max_abs_diff': 1.07e-06, 'greedy_identical': True}
SELFTEST PASSED


In [5]:
run_cmd("verify", ["qwen-verify", "--dtypes", "fp32", "fp16", "--bench",
                   "--out", str(ROOT / "results/internals_qwen_verify.json")], timeout_s=2400)

[fp32] Hugging Face reference
[fp32] own implementation
  manual: max|diff| 1.65e-05 (logits up to 22.0), top-1 agreement 1.000, greedy identical [True, True, True], first divergence [None, None, None], reference top-2 gap there [None, None, None], first NaN layer None
  sdpa: max|diff| 2.10e-05 (logits up to 22.0), top-1 agreement 1.000, greedy identical [True, True, True], first divergence [None, None, None], reference top-2 gap there [None, None, None], first NaN layer None
[fp16] Hugging Face reference
[fp16] own implementation
  manual: max|diff| 4.88e-02 (logits up to 22.0), top-1 agreement 1.000, greedy identical [True, False, True], first divergence [None, 33, None], reference top-2 gap there [None, 0.016, None], first NaN layer None
  sdpa: max|diff| 3.32e-02 (logits up to 22.0), top-1 agreement 1.000, greedy identical [True, False, True], first divergence [None, 33, None], reference top-2 gap there [None, 0.016, None], first NaN layer None
  bench: {'hf_generate': {'prefill_s

In [6]:
run_cmd("lora_peft", ["lora-peft", "--rank", "8",
                      "--out", str(ROOT / "results/internals_lora_peft.json")])

own LoRA
PEFT
  base_vs_adapted: max|diff| 8.54e+00, top-1 agreement 0.909
  own_vs_peft: max|diff| 0.00e+00, top-1 agreement 1.000
  own_unmerged_vs_merged: max|diff| 9.01e-05, top-1 agreement 1.000
  peft_unmerged_vs_merged: max|diff| 9.01e-05, top-1 agreement 1.000
  own_merged_vs_peft_merged: max|diff| 0.00e+00, top-1 agreement 1.000
  trainable params: own 9,232,384 vs PEFT 9,232,384
saved /kaggle/working/llm-lab/results/internals_lora_peft.json


In [7]:
import zipfile

with zipfile.ZipFile("/kaggle/working/llm-lab-exp5.zip", "w", zipfile.ZIP_DEFLATED) as z:
    files = sorted((ROOT / "results").glob("internals_*.json")) + sorted((ROOT / "logs").glob("exp5_*.log"))
    for f in files:
        z.write(f, arcname=f"{f.parent.name}/{f.name}")
print("Zipped:", zipfile.ZipFile("/kaggle/working/llm-lab-exp5.zip").namelist())

Zipped: ['results/internals_lora_peft.json', 'results/internals_qwen_verify.json', 'results/internals_selftest.json', 'logs/exp5_lora_peft.log', 'logs/exp5_selftest.log', 'logs/exp5_verify.log']
